# ML Homework 07: k-NN Classification and K-Means Clustering

## Описание

В этом задании вам предстоит реализовать два фундаментальных алгоритма машинного обучения:
1. **k-NN (k Nearest Neighbors)** — метод k ближайших соседей для классификации
2. **K-Means** — алгоритм кластеризации

Вы научитесь:
1. Работать с метриками расстояния (L1, L2)
2. Реализовывать ленивое обучение (lazy learning)
3. Понимать механику EM-алгоритма на примере K-Means
4. Визуализировать границы решений и результаты кластеризации
5. Использовать PCA и t-SNE для понижения размерности

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_blobs, load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier as SklearnKNN
from sklearn.cluster import KMeans as SklearnKMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, silhouette_score
import ipytest
import pytest
from collections import Counter

%matplotlib inline
plt.style.use('default')

---

## Теоретическая часть

### k-Nearest Neighbors (k-NN)

**Основная идея:** Для классификации нового объекта найдем k ближайших к нему объектов из обучающей выборки и выберем самый популярный класс среди них.

**Метрики расстояния:**
- L1 (Манхэттенское): $d_1(x, x') = \sum_{j=1}^p |x_j - x'_j|$
- L2 (Евклидово): $d_2(x, x') = \sqrt{\sum_{j=1}^p (x_j - x'_j)^2}$

**Алгоритм классификации:**
1. Для каждого тестового объекта вычислить расстояния до всех обучающих объектов
2. Найти k объектов с минимальными расстояниями
3. Провести голосование по классам этих k соседей
4. При ничьей — выбрать класс с минимальной суммой расстояний

**Weighted k-NN:** Каждый сосед голосует с весом $w_i = \frac{1}{d(x, x_i) + \epsilon}$, где $\epsilon$ — малое число для избежания деления на ноль.

### Проклятие размерности (Curse of Dimensionality)

**Проблема:** С ростом числа признаков (размерности) расстояния между точками теряют смысл, и качество k-NN резко падает.

**Математическое объяснение:**

Рассмотрим единичный гиперкуб в $d$-мерном пространстве $[0, 1]^d$:
- **Объем** гиперкуба: $V_d = 1^d = 1$
- **Объем** гипершара радиуса $r$: $V_{sphere} = \frac{\pi^{d/2}}{\Gamma(d/2 + 1)} r^d$

При увеличении размерности $d$:
- Объем гипершара стремится к 0 относительно объема гиперкуба
- Точки становятся "эквидистантными" (равноудаленными друг от друга)
- Расстояние до ближайшего соседа приближается к расстоянию до дальнего

**Последствия для k-NN:**

1. **Потеря различимости:** В высоких размерностях все точки почти одинаково далеко друг от друга
   $$ \lim_{d \to \infty} \frac{\max_i d_i - \min_i d_i}{\min_i d_i} \to 0 $$

2. **Разреженность данных:** Для покрытия пространства требуется экспоненциально больше данных
   - В 1D: 100 точек покрывают отрезок хорошо
   - В 10D: $100^{10}$ точек нужно для аналогичной плотности

3. **Шумовые признаки:** Нерелевантные признаки доминируют над информативными
   - В 10D с 2 информативными признаками: 80% расстояний — шум

**Визуализация эффекта:**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dims = [1, 2, 5, 10, 20, 50, 100]
ratio_stats = []

for d in dims:
    # Генерируем 1000 точек в d-мерном единичном кубе
    points = np.random.rand(1000, d)

    # Для каждой точки находим расстояние до ближайшего и дальнего соседа
    distances = np.zeros((1000, 1000))
    for i in range(1000):
        for j in range(1000):
            distances[i, j] = np.linalg.norm(points[i] - points[j])

    # Исключаем нулевые расстояния (сама точка)
    np.fill_diagonal(distances, np.inf)

    min_d = np.min(distances, axis=1)
    max_d = np.max(distances, axis=1)

    # Отношение max/min показывает "эквидистантность"
    ratio = np.mean(max_d / min_d)
    ratio_stats.append(ratio)

plt.figure(figsize=(10, 6))
plt.plot(dims, ratio_stats, marker="o", linewidth=2)
plt.xlabel("Размерность (d)")
plt.ylabel("Отношение max/min расстояния")
plt.title("Проклятие размерности: все точки становятся равноудаленными")
plt.grid(True)
plt.xscale("log")
plt.show()

**Решения проблемы:**
- **Feature Selection:** оставить только важные признаки
- **Dimensionality Reduction:** PCA, t-SNE, UMAP
- **Distance Weighting:** придать больший вес информативным признакам
- **Нормализация:** масштабировать признаки перед вычислением расстояний
- **Закономерность:** Для k-NN с $n$ обучающими объектами рекомендуется $d \ll \log n$

**Интуиция:** Представьте комнату (3D) — легко найти ближайшего человека. Теперь представьте 100-мерное пространство — понятие "близости" теряет смысл, все точки оказываются примерно на одинаковом расстоянии друг от друга.

### K-Means Clustering

**Основная идея:** Разбить n объектов на k кластеров так, чтобы минимизировать сумму квадратов расстояний от объектов до центроидов их кластеров.

**Целевая функция:**
$$J = \sum_{i=1}^n \sum_{j=1}^k r_{ij} \|x_i - \mu_j\|^2$$

где $r_{ij} = 1$ если объект $i$ принадлежит кластеру $j$, иначе 0.

**Алгоритм:**
1. **Инициализация:** Случайно выбрать k точек как начальные центроиды
2. **E-step (Expectation):** Назначить каждую точку ближайшему центроиду
3. **M-step (Maximization):** Пересчитать центроиды как среднее точек кластера
4. **Повторять** шаги 2-3 пока центроиды не перестанут изменяться или не достигнут max_iterations

### Понижение размерности

**Зачем нужно:**
- Визуализация высокомерных данных
- Ускорение алгоритмов
- Удаление шума
- **Борьба с проклятием размерности**

**Методы:**
- **PCA (Principal Component Analysis):** Линейный метод, находит ортогональные направления максимальной дисперсии
- **t-SNE:** Нелинейный метод, сохраняет локальную структуру данных

В этой лабораторной работе мы будем использовать готовые реализации из sklearn для визуализации данных.

---

## Часть 1: k-NN Классификация

### Задание 1: Метрики расстояния

Реализуйте функции для вычисления L1 (манхэттенского) и L2 (евклидового) расстояний между двумя векторами.

In [ ]:
def l1_distance(x1, x2):
    """
    Вычислить манхэттенское расстояние между двумя векторами.

    d1(x, x') = sum(|xj - x'j|)

    Parameters:
    -----------
    x1, x2 : numpy arrays одинаковой размерности

    Returns:
    --------
    distance : float - манхэттенское расстояние
    """
    #TODO: Реализовать функцию
    return 0.0


def l2_distance(x1, x2):
    """
    Вычислить евклидово расстояние между двумя векторами.

    d2(x, x') = sqrt(sum((xj - x'j)^2))

    Parameters:
    -----------
    x1, x2 : numpy arrays одинаковой размерности

    Returns:
    --------
    distance : float - евклидово расстояние
    """
    #TODO: Реализовать функцию
    return 0.0

### Задание 2: Вычисление матрицы расстояний

Реализуйте функцию для эффективного вычисления матрицы расстояний между всеми парами точек из двух наборов данных используя broadcasting.

In [ ]:
def compute_distances(X_train, X_test, metric="l2"):
    """
    Вычислить матрицу расстояний между всеми парами точек.

    Используйте numpy broadcasting для эффективности.

    Parameters:
    -----------
    X_train : numpy array shape (n_train, n_features)
    X_test : numpy array shape (n_test, n_features)
    metric : 'l1' или 'l2'

    Returns:
    --------
    distances : numpy array shape (n_test, n_train)
                  distances[i, j] = расстояние между X_test[i] и X_train[j]
    """
    #TODO: Реализовать с помощью broadcasting
    # Подсказка: используйте np.linalg.norm для L2 и np.sum(np.abs(...)) для L1
    # Или используйте формулу: ||a-b||^2 = ||a||^2 + ||b||^2 - 2*a*b

    n_test = X_test.shape[0]
    n_train = X_train.shape[0]
    return np.zeros((n_test, n_train))

### Задание 3: Класс KNNClassifier

Реализуйте класс k-NN для классификации с поддержкой:
- Выбора количества соседей (n_neighbors)
- Выбора метрики расстояния (metric)
- Взвешенного голосования (weights)

In [ ]:
class KNNClassifier:
    def __init__(self, n_neighbors=5, metric="l2", weights="uniform"):
        """
        K-Nearest Neighbors классификатор.

        Parameters:
        -----------
        n_neighbors : int - количество соседей (по умолчанию 5)
        metric : 'l1' или 'l2' - метрика расстояния (по умолчанию 'l2')
        weights : 'uniform' или 'distance' - схема взвешивания (по умолчанию 'uniform')
        """
        self.n_neighbors = n_neighbors
        self.metric = metric
        self.weights = weights
        self.X_train = None
        self.y_train = None
        self.classes_ = None

    def fit(self, X, y):
        """
        Обучить модель (lazy learning - просто сохранить данные).

        Parameters:
        -----------
        X : numpy array shape (n_samples, n_features)
        y : numpy array shape (n_samples,)
        """
        #TODO: Сохранить обучающие данные и уникальные классы
        return self

    def predict(self, X):
        """
        Предсказать классы для всех образцов.

        Parameters:
        -----------
        X : numpy array shape (n_samples, n_features)

        Returns:
        --------
        predictions : numpy array shape (n_samples,)
        """
        #TODO: Реализовать предсказание для всех образцов
        raise NotImplementedError

    def predict_proba(self, X):
        """
        Предсказать вероятности классов для всех образцов.

        Parameters:
        -----------
        X : numpy array shape (n_samples, n_features)

        Returns:
        --------
        proba : numpy array shape (n_samples, n_classes)
        """
        #TODO: Реализовать предсказание вероятностей
        raise NotImplementedError

    def _predict_single(self, x):
        """
        Предсказать класс для одного образца.
        """
        #TODO: Реализовать предсказание для одного образца
        # 1. Вычислить расстояния до всех обучающих точек
        # 2. Найти индексы k ближайших соседей (используйте np.argpartition)
        # 3. Получить классы соседей и их расстояния
        # 4. Провести голосование (с учетом весов если weights='distance')
        # 5. При ничьей выбрать класс с минимальной суммой расстояний

        raise NotImplementedError

    def _predict_proba_single(self, x):
        """
        Предсказать вероятности классов для одного образца.
        """
        #TODO: Реализовать вычисление вероятностей
        # Вернуть массив вероятностей для каждого класса
        raise NotImplementedError


### Визуализация границ решений

Функция для визуализации границы принятия решений на 2D данных.

In [ ]:
def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    """
    Визуализировать границу принятия решений для 2D данных.
    """
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), np.arange(y_min, y_max, 0.1))

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap="RdYlBu")
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap="RdYlBu", edgecolor="k", s=50)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(title)
    plt.colorbar(scatter)
    plt.show()

### Тестирование k-NN на синтетических данных

In [ ]:
# Генерация простых 2D данных для визуализации
X_simple, y_simple = make_classification(
    n_samples=200, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=2, random_state=42
)

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_simple, y_simple, test_size=0.3, random_state=42
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

In [ ]:
# Обучение вашей модели
knn = KNNClassifier(n_neighbors=5, metric='l2', weights='uniform')
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print(f"Accuracy (My KNN): {accuracy_score(y_test, y_pred):.4f}")

# Сравнение со sklearn
sk_knn = SklearnKNN(n_neighbors=5)
sk_knn.fit(X_train, y_train)
y_pred_sk = sk_knn.predict(X_test)

print(f"Accuracy (Sklearn): {accuracy_score(y_test, y_pred_sk):.4f}")

In [ ]:
# Визуализация границы решений
plot_decision_boundary(knn, X_train, y_train, f"KNN Decision Boundary (k={knn.n_neighbors})")

In [ ]:
# Сравнение uniform vs distance weights
knn_uniform = KNNClassifier(n_neighbors=5, weights='uniform')
knn_distance = KNNClassifier(n_neighbors=5, weights='distance')

knn_uniform.fit(X_train, y_train)
knn_distance.fit(X_train, y_train)

acc_uniform = accuracy_score(y_test, knn_uniform.predict(X_test))
acc_distance = accuracy_score(y_test, knn_distance.predict(X_test))

print(f"Accuracy (uniform weights): {acc_uniform:.4f}")
print(f"Accuracy (distance weights): {acc_distance:.4f}")

### Тесты (ipytest)

In [ ]:
ipytest.autoconfig()


class TestKNN:
    def test_l1_distance(self):
        """Тест L1 расстояния"""
        x1 = np.array([1, 2, 3])
        x2 = np.array([2, 3, 4])
        expected = 3  # |1-2| + |2-3| + |3-4| = 1 + 1 + 1 = 3
        assert l1_distance(x1, x2) == expected

    def test_l2_distance(self):
        """Тест L2 расстояния"""
        x1 = np.array([0, 0])
        x2 = np.array([3, 4])
        expected = 5  # sqrt(3^2 + 4^2) = 5
        assert np.isclose(l2_distance(x1, x2), expected)

    def test_simple_classification(self):
        """Тест классификации на простых данных"""
        # Два хорошо разделенных кластера
        X = np.array(
            [
                [0, 0],
                [0, 1],
                [1, 0],
                [1, 1],  # класс 0
                [10, 10],
                [10, 11],
                [11, 10],
                [11, 11],
            ]
        )  # класс 1
        y = np.array([0, 0, 0, 0, 1, 1, 1, 1])

        knn = KNNClassifier(n_neighbors=3, metric="l2")
        knn.fit(X, y)

        # Тестируем точки рядом с кластерами
        assert knn._predict_single(np.array([0.5, 0.5])) == 0
        assert knn._predict_single(np.array([10.5, 10.5])) == 1

    def test_tie_breaking(self):
        """Тест разрывания связей"""
        # Точка равноудалена от двух классов
        X = np.array(
            [
                [0, 0],
                [0, 2],  # класс 0
                [2, 0],
                [2, 2],
            ]
        )  # класс 1
        y = np.array([0, 0, 1, 1])

        knn = KNNClassifier(n_neighbors=2, metric="l2")
        knn.fit(X, y)

        # Точка [1, 1] равноудалена от всех четырех точек
        # Должна выбрать класс с минимальной суммой расстояний
        prediction = knn._predict_single(np.array([1, 1]))
        assert prediction in [0, 1]  # Должен выбрать один из классов

    def test_predict_proba(self):
        """Тест предсказания вероятностей"""
        X = np.array(
            [
                [0, 0],
                [0, 1],  # класс 0
                [10, 10],
                [10, 11],
            ]
        )  # класс 1
        y = np.array([0, 0, 1, 1])

        knn = KNNClassifier(n_neighbors=2, metric="l2")
        knn.fit(X, y)

        proba = knn._predict_proba_single(np.array([0.5, 0.5]))
        assert len(proba) == 2  # Два класса
        assert np.isclose(proba.sum(), 1.0)  # Сумма вероятностей = 1
        assert proba[0] > proba[1]  # Ближе к классу 0


ipytest.run()

---

## Часть 2: Понижение размерности

В этой части мы используем готовые реализации PCA и t-SNE из sklearn для визуализации данных перед кластеризацией.

In [ ]:
# Загрузка датасета Iris (4 признака, 3 класса)
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"Iris dataset shape: {X_iris.shape}")
print(f"Features: {feature_names}")
print(f"Classes: {target_names}")

In [ ]:
# Применение PCA
pca = PCA(n_components=2)
X_iris_pca = pca.fit_transform(X_iris)

# Применение t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_iris_tsne = tsne.fit_transform(X_iris)

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# PCA
scatter1 = axes[0].scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=y_iris, cmap='viridis', s=50)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('PCA Visualization')
plt.colorbar(scatter1, ax=axes[0], label='Class')

# t-SNE
scatter2 = axes[1].scatter(X_iris_tsne[:, 0], X_iris_tsne[:, 1], c=y_iris, cmap='viridis', s=50)
axes[1].set_xlabel('Component 1')
axes[1].set_ylabel('Component 2')
axes[1].set_title('t-SNE Visualization')
plt.colorbar(scatter2, ax=axes[1], label='Class')

plt.tight_layout()
plt.show()

print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.4f}")

In [ ]:
# График explained variance для разного числа компонент
pca_full = PCA()
pca_full.fit(X_iris)

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(pca_full.explained_variance_ratio_) + 1), 
         np.cumsum(pca_full.explained_variance_ratio_),
         marker='o', linestyle='--')
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance')
plt.title('Explained Variance vs Number of Components')
plt.grid(True)
plt.axhline(y=0.95, color='r', linestyle=':', label='95% variance')
plt.legend()
plt.show()

---

## Часть 3: K-Means Кластеризация

### Задание 4: Класс KMeansClustering

Реализуйте алгоритм K-Means с нуля.

In [ ]:
class KMeansClustering:
    def __init__(self, n_clusters=3, max_iterations=100, tol=1e-4, random_state=None):
        """
        K-Means кластеризация.

        Parameters:
        -----------
        n_clusters : int - количество кластеров
        max_iterations : int - максимальное число итераций
        tol : float - порог сходимости (изменение центроидов)
        random_state : int - для воспроизводимости
        """
        self.n_clusters = n_clusters
        self.max_iterations = max_iterations
        self.tol = tol
        self.random_state = random_state
        self.centroids = None
        self.labels_ = None
        self.inertia_ = None
        self.history = []  # Для сохранения истории центроидов

    def fit(self, X):
        """
        Обучить модель кластеризации.

        Parameters:
        -----------
        X : numpy array shape (n_samples, n_features)
        """
        #TODO: Реализовать основной алгоритм
        # 1. Инициализировать центроиды
        # 2. Повторять max_iterations раз:
        #    a. Назначить точки ближайшим центроидам (E-step)
        #    b. Пересчитать центроиды как среднее (M-step)
        #    c. Проверить сходимость
        # 3. Вычислить финальную inertia

        raise NotImplementedError

    def predict(self, X):
        """
        Назначить кластеры для новых данных.

        Parameters:
        -----------
        X : numpy array shape (n_samples, n_features)

        Returns:
        --------
        labels : numpy array shape (n_samples,)
        """
        #TODO: Реализовать предсказание кластеров
        raise NotImplementedError

    def _initialize_centroids(self, X):
        """
        Инициализировать центроиды случайным выбором из данных.
        """
        #TODO: Реализовать случайную инициализацию
        raise NotImplementedError

    def _assign_clusters(self, X):
        """
        E-step: назначить каждую точку ближайшему центроиду.
        """
        #TODO: Реализовать назначение кластеров
        # Подсказка: вычислить расстояния до всех центроидов, выбрать минимальный
        raise NotImplementedError

    def _update_centroids(self, X, labels):
        """
        M-step: обновить центроиды как среднее точек кластера.
        """
        #TODO: Реализовать обновление центроидов
        # Обработка пустых кластеров: сохранить старый центроид
        raise NotImplementedError

    def _compute_inertia(self, X):
        """
        Вычислить сумму квадратов расстояний до ближайших центроидов.
        """
        #TODO: Реализовать вычисление inertia
        raise NotImplementedError


### Визуализация K-Means

In [ ]:
def plot_kmeans_result(X, centroids, labels, title="K-Means Clustering"):
    """
    Визуализировать результаты кластеризации для 2D данных.
    """
    plt.figure(figsize=(10, 8))

    # Отобразить точки с цветами кластеров
    scatter = plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="viridis", s=50, alpha=0.6)

    # Отобразить центроиды
    plt.scatter(centroids[:, 0], centroids[:, 1], c="red", marker="x", s=200, linewidths=3, label="Centroids")

    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(title)
    plt.legend()
    plt.colorbar(scatter)
    plt.show()

In [ ]:
# Генерация данных для кластеризации
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=1.5, random_state=42)

print(f"Blobs dataset shape: {X_blobs.shape}")

In [ ]:
# Обучение вашей модели
kmeans = KMeansClustering(n_clusters=3, random_state=42)
kmeans.fit(X_blobs)

print(f"K-Means converged after {len(kmeans.history)} iterations")
print(f"Final inertia: {kmeans.inertia_:.4f}")

# Визуализация
plot_kmeans_result(X_blobs, kmeans.centroids, kmeans.labels_, 
                  f"K-Means Clustering (k={kmeans.n_clusters})")

In [ ]:
# Сравнение со sklearn
sk_kmeans = SklearnKMeans(n_clusters=3, random_state=42, n_init=10)
sk_kmeans.fit(X_blobs)

print(f"My K-Means inertia: {kmeans.inertia_:.4f}")
print(f"Sklearn inertia: {sk_kmeans.inertia_:.4f}")

### Метод локтя для выбора k

In [ ]:
def plot_elbow_method(X, max_k=10, random_state=42):
    """
    Построить график метода локтя для выбора оптимального k.
    """
    inertias = []
    K_range = range(1, max_k + 1)

    for k in K_range:
        kmeans = KMeansClustering(n_clusters=k, random_state=random_state)
        kmeans.fit(X)
        inertias.append(kmeans.inertia_)

    plt.figure(figsize=(10, 6))
    plt.plot(K_range, inertias, marker="o", linestyle="--")
    plt.xlabel("Number of clusters (k)")
    plt.ylabel("Inertia")
    plt.title("Elbow Method for Optimal k")
    plt.grid(True)
    plt.show()

    return inertias

In [ ]:
# Применение метода локтя
inertias = plot_elbow_method(X_blobs, max_k=10, random_state=42)

### Тесты (ipytest)

In [ ]:
class TestKMeans:
    def test_simple_clustering(self):
        """Тест на хорошо разделенных кластерах"""
        # Три хорошо разделенных кластера
        X = np.vstack(
            [
                np.random.randn(50, 2) + [0, 0],
                np.random.randn(50, 2) + [10, 0],
                np.random.randn(50, 2) + [0, 10],
            ]
        )

        kmeans = KMeansClustering(n_clusters=3, random_state=42)
        kmeans.fit(X)

        assert kmeans.centroids.shape == (3, 2)
        assert len(kmeans.labels_) == 150
        assert kmeans.inertia_ > 0

    def test_convergence(self):
        """Тест сходимости алгоритма"""
        X = np.random.randn(100, 2)

        kmeans = KMeansClustering(n_clusters=3, max_iterations=50, random_state=42)
        kmeans.fit(X)

        # Проверить, что история содержит не более max_iterations + 1 записей
        assert len(kmeans.history) <= 51  # +1 за начальную инициализацию

    def test_predict_new_data(self):
        """Тест предсказания на новых данных"""
        X = np.vstack([np.random.randn(50, 2) + [0, 0], np.random.randn(50, 2) + [10, 10]])

        kmeans = KMeansClustering(n_clusters=2, random_state=42)
        kmeans.fit(X)

        # Предсказываем для новых точек
        X_new = np.array([[0, 0], [10, 10], [5, 5]])
        labels = kmeans.predict(X_new)

        assert len(labels) == 3
        assert labels[0] == labels[1]  # Должны быть в одном кластере или нет

    def test_inertia_monotonic(self):
        """Тест монотонного убывания inertia"""
        X = np.random.randn(100, 2)

        kmeans = KMeansClustering(n_clusters=3, random_state=42)
        kmeans.fit(X)

        # Inertia должна монотонно убываться
        inertias = []
        for centroids in kmeans.history:
            # Вычисляем inertia для данных центроидов
            temp_kmeans = KMeansClustering(n_clusters=3)
            temp_kmeans.centroids = centroids
            temp_kmeans.labels_ = temp_kmeans._assign_clusters(X)
            inertias.append(temp_kmeans._compute_inertia(X))

        for i in range(1, len(inertias)):
            assert inertias[i] <= inertias[i - 1] + 1e-10  # Допускаем небольшую погрешность


ipytest.run()

---

## Часть 4: Сравнение со Sklearn

### Задание 5: Исследование k-NN

Исследуйте влияние гиперпараметра k на качество классификации.

In [ ]:
# Генерация более сложного датасета
X_complex, y_complex = make_classification(
    n_samples=500, n_features=10, n_informative=8, n_redundant=2,
    n_clusters_per_class=2, random_state=42
)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_complex, y_complex, test_size=0.3, random_state=42
)

print(f"Train: {X_train_c.shape}, Test: {X_test_c.shape}")

In [ ]:
#TODO: Исследовать accuracy для разных значений k
k_values = [1, 3, 5, 7, 11, 15, 21, 31]
train_acc = []
test_acc = []

for k in k_values:
    #TODO: Обучить вашу модель KNNClassifier и sklearn
    # Вычислить accuracy на train и test
    pass

# Визуализация
plt.figure(figsize=(10, 6))
plt.plot(k_values, train_acc, marker='o', label='Train Accuracy')
plt.plot(k_values, test_acc, marker='o', label='Test Accuracy')
plt.xlabel('Number of neighbors (k)')
plt.ylabel('Accuracy')
plt.title('KNN: Accuracy vs k')
plt.legend()
plt.grid(True)
plt.show()

#TODO: Вывести лучшее значение accuracy и соответствующий k


### Задание 6: Реальные датасеты

Протестируйте ваши реализации на реальных датасетах.

In [ ]:
# K-NN на Iris датасете
iris = load_iris()
X_iris_full = iris.data
y_iris_full = iris.target

X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris_full, y_iris_full, test_size=0.3, random_state=42
)

#TODO: Обучить ваш KNNClassifier и сравнить со sklearn
# 1. Создать две модели
# 2. Обучить их на train
# 3. Посчитать accuracy на test
# 4. Вывести результаты


In [ ]:
# K-Means на Wine датасете (unsupervised)
wine = load_wine()
X_wine = wine.data

#TODO: Применить ваш KMeansClustering и сравнить со sklearn
# Используйте silhouette_score для оценки качества
from sklearn.metrics import silhouette_score

#TODO: 1. Создать и обучить две модели
#TODO: 2. Вычислить inertia и silhouette_score
#TODO: 3. Вывести сравнение результатов


---

## Итоговые тесты

In [ ]:
class TestFinal:
    def test_knn_vs_sklearn_accuracy(self):
        """Тест: ваш KNN должен быть близок к sklearn по качеству"""
        X, y = make_classification(n_samples=200, n_features=5, random_state=42)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

        my_knn = KNNClassifier(n_neighbors=5, metric="l2")
        sk_knn = SklearnKNN(n_neighbors=5)

        my_knn.fit(X_train, y_train)
        sk_knn.fit(X_train, y_train)

        my_acc = accuracy_score(y_test, my_knn.predict(X_test))
        sk_acc = accuracy_score(y_test, sk_knn.predict(X_test))

        # Допускаем отклонение не более 5%
        assert abs(my_acc - sk_acc) < 0.05, (
            f"Ваш KNN сильно отличается от sklearn: {my_acc:.4f} vs {sk_acc:.4f}"
        )

    def test_kmeans_converges(self):
        """Тест: K-Means должен сходиться"""
        X, _ = make_blobs(n_samples=200, centers=4, random_state=42)

        kmeans = KMeansClustering(n_clusters=4, max_iterations=100, random_state=42)
        kmeans.fit(X)

        # Проверить, что нашли именно 4 кластера
        unique_labels = np.unique(kmeans.labels_)
        assert len(unique_labels) <= 4, "Найдено больше кластеров, чем ожидается"

    def test_weighted_knn(self):
        """Тест взвешенного k-NN"""
        X = np.array(
            [
                [0, 0],
                [0.1, 0],  # класс 0, близко к тестовой точке
                [10, 10],
                [10, 11],  # класс 1, далеко
            ]
        )
        y = np.array([0, 0, 1, 1])

        knn_uniform = KNNClassifier(n_neighbors=3, weights="uniform")
        knn_distance = KNNClassifier(n_neighbors=3, weights="distance")

        knn_uniform.fit(X, y)
        knn_distance.fit(X, y)

        # Точка [0.05, 0] ближе к классу 0
        test_point = np.array([0.05, 0])

        pred_uniform = knn_uniform._predict_single(test_point)
        pred_distance = knn_distance._predict_single(test_point)

        # Оба должны предсказать класс 0
        assert pred_uniform == 0, f"Uniform KNN должен предсказать 0, получил {pred_uniform}"
        assert pred_distance == 0, f"Distance KNN должен предсказать 0, получил {pred_distance}"


ipytest.run()

---

## Заключение

### Выводы по лабораторной работе:

**k-NN (k Nearest Neighbors):**
- Простой и интуитивно понятный алгоритм
- Lazy learning: не требует этапа обучения
- Чувствителен к выбору метрики расстояния
- Weighted voting может улучшить качество на зашумленных данных
- Выбор k важен: малые k → переобучение, большие k → недообучение

**K-Means:**
- Простой EM-алгоритм для кластеризации
- Требует предварительного указания числа кластеров
- Чувствителен к инициализации центроидов
- Метод локтя помогает выбрать оптимальное k
- Inertia монотонно убывает с ростом k

**Понижение размерности:**
- PCA: линейный метод, быстрый, сохраняет глобальную структуру
- t-SNE: нелинейный метод, медленный, сохраняет локальную структуру
- Оба метода полезны для визуализации высокомерных данных
